# 🧪 Sesión 13 · Manejo de Sesiones Stateful vs Stateless Tokens

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo recuerda el servidor que ya iniciaste sesión?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/login  {"usuario":"ana","password":"secreta"}
HTTP/1.1 200 OK
Set-Cookie: session_id=...; HttpOnly; Path=/; SameSite=Lax
{"mensaje":"sesión iniciada"}

GET /api/perfil   (con la cookie)
HTTP/1.1 200 OK
{"usuario":"ana"}

POST /api/logout  (con la cookie)
HTTP/1.1 204 No Content

GET /api/perfil   (después del logout)
HTTP/1.1 401 Unauthorized
```

_La cookie viaja sola: el cliente no tiene que copiar el identificador en cada petición._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar una sesión con estado en el servidor.
- Emitir una cookie segura y leerla en peticiones posteriores.
- Diferenciar sesión con estado de un token sin estado.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Login correcto y cookie HttpOnly → HTTP 200
- [ ] El perfil se obtiene con la cookie → HTTP 200
- [ ] El logout cierra la sesión → HTTP 204
- [ ] Después del logout el perfil es 401 → HTTP 401

## 🧩 El reto

Implementa login, perfil y logout usando sesiones en memoria y una cookie HttpOnly.

**Requisitos del contrato**

- El usuario de prueba es ana con contraseña secreta.
- POST /api/login valida usuario y contraseña y responde 200.
- La respuesta de login incluye una cookie session_id con HttpOnly y SameSite=Lax.
- GET /api/perfil devuelve el usuario si la cookie es válida; si no, 401.
- POST /api/logout invalida la sesión y responde 204.
- Después del logout, /api/perfil vuelve a responder 401.

> **Pista conceptual:** El cliente solo guarda un identificador opaco. Todo lo demás (usuario, expiración, permisos) vive en el servidor.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion13.md` con la diferencia entre sesión y token.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py

from fastapi import FastAPI, HTTPException, Response, Cookie, status
from pydantic import BaseModel
import secrets

app = FastAPI(title="Sistema de Sesiones Stateful"

USUARIOS = {
    "ana": "secreta"
}

sesiones = {}


class LoginEntrada(BaseModel)
    usuario: str
    password: str


@app.post("/api/login")
def login(datos: LoginEntrada, response: Response)

    if USUARIOS.get(datos.usuario) != datos.password
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Credenciales incorrectas"
        )

    session_id = secrets.token_urlsafe(32)

    sesiones[session_id] = datos.usuario

    response.set_cookie(
        key="session_id",
        value=session_id,
        httponly=True,
        samesite="lax",
        path="/"
    )

    return {"mensaje": "sesión iniciada"}


@app.get("/api/perfil")
def perfil(session_id: str = Cookie(default=None)):

    usuario = sesiones.get(session_id)

    if usuario is None
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Sesión no válida"
        )

    return {"usuario": usuario}


@app.post("/api/logout", status_code=status.HTTP_204_NO_CONTENT)
def logout(
    response: Response,
    session_id: str = Cookie(default=None)
)

    if session_id:
        sesiones.pop(session_id, None)

    response.delete_cookie(
        key="session_id",
        path="/",
        httponly=True,
        samesite="lax"
    )

    return response


Writing app.py


In [6]:


%%writefile app.py

from fastapi import FastAPI, HTTPException, Response, Cookie, status
from pydantic import BaseModel
import secrets

app = FastAPI(title="Sistema de Sesiones Stateful")

USUARIOS = {
    "ana": "secreta"
}

sesiones = {}


class LoginEntrada(BaseModel):
    usuario: str
    password: str


@app.post("/api/login")
def login(datos: LoginEntrada, response: Response):

    if USUARIOS.get(datos.usuario) != datos.password:
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas"
        )

    session_id = secrets.token_urlsafe(32)

    sesiones[session_id] = datos.usuario

    response.set_cookie(
        key="session_id",
        value=session_id,
        httponly=True,
        samesite="lax",
        path="/"
    )

    return {"mensaje": "sesión iniciada"}


@app.get("/api/perfil")
def perfil(session_id: str = Cookie(default=None)):

    usuario = sesiones.get(session_id)

    if usuario is None:
        raise HTTPException(
            status_code=401,
            detail="Sesión no válida"
        )

    return {"usuario": usuario}


@app.post("/api/logout", status_code=204)
def logout(
    session_id: str = Cookie(default=None)
):

    if session_id:
        sesiones.pop(session_id, None)

    return Response(
        status_code=204,
        headers={
            "Set-Cookie": "session_id=; HttpOnly; Path=/; SameSite=Lax; Max-Age=0"
        }
    )



Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Estoy trabajando con Python y FastAPI en Google Colab. El entorno utiliza FastAPI, Pydantic y TestClient para desarrollar y comprobar una API REST. La práctica consiste en implementar un sistema de sesiones Stateful utilizando almacenamiento en memoria y cookies HttpOnly. |
| **Rol** | Actúa como un desarrollador especializado en Python, FastAPI y diseño de API REST. Tu función es ayudarme a implementar un sistema de autenticación mediante sesiones, respetando los requisitos establecidos y utilizando código claro, funcional y fácil de comprender. |
| **Especificación** | Desarrollar una API que permita iniciar sesión, consultar el perfil del usuario y cerrar sesión. |
| **Output** | Entregar el archivo app.py completo y funcional, utilizando Python y FastAPI. El resultado debe cumplir los cuatro criterios de aceptación del verificador oficial y permitir comprobar el inicio de sesión, la consulta del perfil, el cierre de sesión y el rechazo del acceso después del logout. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [7]:
from verificar import probar_http

casos = [{'descripcion': 'Login correcto y cookie HttpOnly',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'headers_contiene': {'set-cookie': ['httponly', 'samesite']}},
 {'descripcion': 'El perfil se obtiene con la cookie',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'estado': 200,
  'json_contiene': {'usuario': 'ana'}},
 {'descripcion': 'El logout cierra la sesión', 'metodo': 'POST', 'ruta': '/api/logout', 'estado': 204},
 {'descripcion': 'Después del logout el perfil es 401', 'metodo': 'GET', 'ruta': '/api/perfil', 'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. Login correcto y cookie HttpOnly
✅ 02. El perfil se obtiene con la cookie
✅ 03. El logout cierra la sesión
✅ 04. Después del logout el perfil es 401
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion13.md` a `auditorias/auditoria_sesion13.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue conseguir que el servidor reconociera al usuario después del inicio de sesión. Tuve que utilizar una cookie para conservar el identificador de sesión y comprobar que todavía existiera en el servidor.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?Decidí utilizar secrets.token_urlsafe() para generar identificadores de sesión aleatorios y evitar utilizar identificadores fáciles de adivinar. También organicé las sesiones mediante un diccionario en memoria.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   La verificación me permitió comprobar que no era suficiente con cerrar la sesión en el navegador. También era necesario eliminar el identificador del almacenamiento del servidor para impedir que se pudiera volver a utilizar.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
